# 02 · Data Ingestion — UN Comtrade (imports by country of origin)

**Goal:** pull import volumes, by country of origin, for EVs, hybrids, combustion pickups,
lithium-ion batteries, and general auto parts — across 8 Latin American countries. This is
the dataset that supports the "Chinese brands vs. Japanese/Korean vs. US vs. European" market
share analysis, and the "spare parts opportunity" angle (via the lithium battery HS code).

**Fixes applied in this version (v4):**

1. **Country of origin was empty in v2.** The Comtrade preview endpoint's `partnerDesc` text
   field is frequently blank on the free/keyless tier. The fix: resolve `partnerCode` (which
   *is* reliably populated) against Comtrade's own official reference table
   (`get_comtrade_partner_lookup()` in `src/utils.py`), instead of trusting the text field.
2. **~35% duplicate rows in v2.** `save_processed()` now deduplicates by default (see
   `src/utils.py`) and reports how many rows were dropped.
3. **"World" was being counted as a country of origin (v3).** Even without passing
   `partnerCode=0`, the preview endpoint still returns an aggregate "World" row (`Origin_Code
   == '0'`) alongside the real per-country breakdown. Left in, this silently doubles totals —
   confirmed by checking Brazil 2023 EV imports, where "World" ($796M) was nearly identical to
   the sum of all real countries of origin for that row. This version splits it into its own
   `comtrade_world_totals_latam.csv` file and uses it as a cross-check for truncated queries
   (see step 5b), instead of mixing it into the per-country data.

**HS codes used:**

| Code | Description |
|---|---|
| 8703.80 | Battery electric vehicles (BEV) |
| 8703.40 | Hybrid vehicles (HEV/PHEV) |
| 8704.21 | Light-duty combustion pickups |
| 8507.60 | Lithium-ion batteries |
| 8708 | General auto parts and accessories |

In [11]:
import sys, time, os
sys.path.append('../src')
from utils import call_json_api, save_processed, get_comtrade_partner_lookup, DATA_RAW
import pandas as pd

# Optional: if you register for a free subscription key at https://comtradedeveloper.un.org/,
# set it here to raise the per-query cap from 500 to 100,000 records.
COMTRADE_KEY = os.environ.get("COMTRADE_SUBSCRIPTION_KEY", "")

In [2]:
REPORTERS = {
    'Peru': '604', 'Brazil': '076', 'Mexico': '484', 'Chile': '152',
    'Colombia': '170', 'Argentina': '032', 'Ecuador': '218', 'Bolivia': '068',
}

HS_CODES = {
    'EV_100pct':          '870380',
    'Hybrid':             '870340',
    'Combustion_pickup':  '870421',
    'Lithium_battery':    '850760',
    'General_auto_parts': '8708',
}

# Country -> region/brand-bloc mapping, used later for the "who's winning" analysis.
# Extend this if the coverage check (below) shows meaningful unclassified volume.
REGION_BY_COUNTRY = {
    'China': 'China',
    'Japan': 'Japan/Korea', 'Rep. of Korea': 'Japan/Korea', 'Korea': 'Japan/Korea',
    'USA': 'USA/Mexico', 'United States': 'USA/Mexico', 'Mexico': 'USA/Mexico',
    'Germany': 'Europe', 'France': 'Europe', 'Spain': 'Europe', 'Italy': 'Europe',
    'Czechia': 'Europe', 'Slovakia': 'Europe', 'Sweden': 'Europe', 'Belgium': 'Europe',
    'India': 'India', 'Thailand': 'Southeast Asia', 'Indonesia': 'Southeast Asia',
    'Brazil': 'LatAm (re-export)', 'Argentina': 'LatAm (re-export)',
}

## 1. Load the official Comtrade country reference table

Downloaded once and cached locally in `data/raw/`. This is what resolves `Origen_Code` into
a country name — not the (often-blank) `partnerDesc` field from the trade data itself.

In [3]:
partner_lookup = get_comtrade_partner_lookup()
print(f"{len(partner_lookup)} reference entries loaded.")
# Spot check: code 156 should resolve to China
print("Code 156 ->", partner_lookup.get('156', 'NOT FOUND'))

16:45:54 | INFO | Loaded 310 partner-country reference entries.


310 reference entries loaded.
Code 156 -> China


## 2. Query function

We deliberately do NOT pass `partnerCode=0` (which would aggregate all trade partners into a
single "World" row). Omitting it returns one row per partner country actually reported for
that reporter/HS/year combination — typically well under the 500-row cap of the free tier for
this scale of query.

In [4]:
def query_comtrade(reporter_code: str, hs_code: str, period: str, flow: str = 'M') -> list:
    """
    Query the Comtrade preview endpoint for one reporter/commodity/period, broken
    down by trading partner. Returns the raw list of result dicts.
    """
    base_url = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
    params = {
        'reporterCode': reporter_code,
        'cmdCode': hs_code,
        'flowCode': flow,
        'period': period,
    }
    try:
        data = call_json_api(base_url, params=params, retries=2)
        rows = data.get('data', [])
        if len(rows) >= 500:
            print(f"  [ALERT] {reporter_code}/{hs_code}/{period} returned {len(rows)} rows "
                  "-- may be truncated by the free-tier cap. Review manually.")
        return rows
    except Exception as e:
        print(f"  [WARN] failed {reporter_code}/{hs_code}/{period}: {e}")
        return []

## 3. Run the extraction

8 countries × 5 categories × 5 years = 200 calls, ~1 request/second ≈ 3–4 minutes. This is
the cost of the free, keyless tier — normal and expected.

In [5]:
YEARS = ['2021', '2022', '2023', '2024', '2025']

results = []
total_calls = len(REPORTERS) * len(HS_CODES) * len(YEARS)
counter = 0

for country, rep_code in REPORTERS.items():
    for category, hs in HS_CODES.items():
        for year in YEARS:
            counter += 1
            print(f"[{counter}/{total_calls}] {country} | {category} | {year}", end='\r')
            rows = query_comtrade(rep_code, hs, year)
            for row in rows:
                origin_code = str(row.get('partnerCode'))
                results.append({
                    'Country': country,
                    'Category': category,
                    'HS_Code': hs,
                    'Year': row.get('period'),
                    'Origin_Code': origin_code,
                    # Resolved from the reference table, NOT the raw text field.
                    'Origin': partner_lookup.get(origin_code, row.get('partnerDesc') or 'Unknown'),
                    'CIF_Value_USD': row.get('primaryValue'),
                    'Net_Weight_Kg': row.get('netWgt'),
                    'Quantity': row.get('qty'),
                })
            time.sleep(1)

print(f"\nDone. {len(results)} raw records collected.")

16:46:20 | WARNING | Attempt 1 on https://comtradeapi.un.org/public/v1/preview/C/A/HS failed: 429 Client Error: Too Many Requests for url: https://comtradeapi.un.org/public/v1/preview/C/A/HS?reporterCode=604&cmdCode=870421&flowCode=M&period=2021


  [ALERT] 604/850760/2024 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 604/8708/2024 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/850760/2021 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/850760/2022 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/850760/2023 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/850760/2024 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/850760/2025 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/8708/2021 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/8708/2022 returned 500 rows -- may be truncated by the free-tier cap. Review manually.
  [ALERT] 170/8708/2023 returned 500 rows -- may be truncated by the free-tier 

In [6]:
df_comtrade = pd.DataFrame(results)

# The Comtrade preview endpoint returns a "World" aggregate row (Origin_Code == '0')
# ALONGSIDE the individual-partner breakdown, even without passing partnerCode=0
# explicitly. This is not an extra country — it's the official total for that
# reporter/category/year, and summing it together with the individual countries
# double-counts the total. We split it into its own reference series and keep
# only real countries in the per-origin breakdown used for analysis.
is_world_total = df_comtrade['Origin_Code'] == '0'
df_world_totals = df_comtrade[is_world_total].drop(columns=['Origin_Code', 'Origin'])
df_comtrade = df_comtrade[~is_world_total].reset_index(drop=True)

print(f"Split off {len(df_world_totals)} 'World' aggregate rows into a separate reference series.")
print(f"Remaining rows (real countries of origin only): {len(df_comtrade)}")

# Quick check: how much of the origin field actually resolved?
unresolved = (df_comtrade['Origin'] == 'Unknown').sum()
print(f"Origin resolved for {len(df_comtrade) - unresolved} / {len(df_comtrade)} rows "
      f"({unresolved} unresolved — likely rare/obsolete country codes, safe to ignore if small).")

df_comtrade.head(10)

Split off 3007 'World' aggregate rows into a separate reference series.
Remaining rows (real countries of origin only): 25981
Origin resolved for 25981 / 25981 rows (0 unresolved — likely rare/obsolete country codes, safe to ignore if small).


,Country,Category,HS_Code,Year,Origin_Code,Origin,CIF_Value_USD,Net_Weight_Kg,Quantity
0,Peru,EV_100pct,870380,2021,842,USA,46052.120,1770.00,1.0
1,Peru,EV_100pct,870380,2021,56,Belgium,679877.480,30823.00,12.0
2,Peru,EV_100pct,870380,2021,410,Rep. of Korea,249041.250,13680.00,9.0
3,Peru,EV_100pct,870380,2021,276,Germany,588490.100,14592.00,6.0
4,Peru,EV_100pct,870380,2021,156,China,237039.940,34450.74,90.0
5,Peru,EV_100pct,870380,2022,56,Belgium,1917868.551,83842.00,32.0
6,Peru,EV_100pct,870380,2022,410,Rep. of Korea,1407352.880,73517.00,47.0
7,Peru,EV_100pct,870380,2022,156,China,508273.574,38735.10,53.0
8,Peru,EV_100pct,870380,2022,826,United Kingdom,426888.232,19542.20,14.0
9,Peru,EV_100pct,870380,2022,276,Germany,416471.423,12118.00,5.0


## 4. Map country of origin to region/brand bloc

In [7]:
df_comtrade['Origin_Region'] = df_comtrade['Origin'].map(REGION_BY_COUNTRY).fillna('Other/Unclassified')

unclassified = df_comtrade[df_comtrade['Origin_Region'] == 'Other/Unclassified']
if len(unclassified) > 0:
    print("Unclassified origins and their total USD value (consider adding high-value ones to REGION_BY_COUNTRY):")
    print(unclassified.groupby('Origin')['CIF_Value_USD'].sum().sort_values(ascending=False).head(15))

Unclassified origins and their total USD value (consider adding high-value ones to REGION_BY_COUNTRY):
Origin
Canada             1.591066e+10
Poland             1.271496e+10
Hungary            3.951283e+09
Uruguay            3.330005e+09
Viet Nam           2.851059e+09
Türkiye            2.691773e+09
Malaysia           2.581476e+09
Romania            2.419603e+09
Other Asia, nes    2.275643e+09
Austria            1.562786e+09
United Kingdom     1.526403e+09
Kenya              1.383258e+09
Netherlands        1.053371e+09
Areas, nes         7.550647e+08
Chile              6.563573e+08
Name: CIF_Value_USD, dtype: float64


## 5. Final quality check and save

In [8]:
coverage = df_comtrade.groupby(['Country', 'Category'])['Year'].nunique().reset_index()
coverage.columns = ['Country', 'Category', 'years_with_data']
coverage.sort_values('years_with_data')

,Country,Category,years_with_data
39,Peru,Lithium_battery,4
35,Peru,Combustion_pickup,4
38,Peru,Hybrid,4
37,Peru,General_auto_parts,4
36,Peru,EV_100pct,4
21,Colombia,EV_100pct,5
22,Colombia,General_auto_parts,5
23,Colombia,Hybrid,5
24,Colombia,Lithium_battery,5
25,Ecuador,Combustion_pickup,5


### 5b. Truncation check using the official "World" total

Some queries hit the 500-row cap during extraction (flagged by the `[ALERT]` messages
printed above, if any appeared). We can cross-check this systematically: if the sum of our
individual-country rows is meaningfully below Comtrade's own official "World" total for the
same reporter/category/year, that combination was very likely truncated and is missing some
countries of origin. Flag anything off by more than 10% for manual review.

In [9]:
sum_by_group = df_comtrade.groupby(['Country', 'Category', 'Year'])['CIF_Value_USD'].sum().reset_index()
world_by_group = df_world_totals.groupby(['Country', 'Category', 'Year'])['CIF_Value_USD'].sum().reset_index()

check = sum_by_group.merge(world_by_group, on=['Country', 'Category', 'Year'], suffixes=('_summed', '_world_total'))
check['pct_of_official_total'] = (check['CIF_Value_USD_summed'] / check['CIF_Value_USD_world_total'] * 100).round(1)

likely_truncated = check[check['pct_of_official_total'] < 90].sort_values('pct_of_official_total')
if len(likely_truncated) > 0:
    print(f"{len(likely_truncated)} country/category/year combinations look truncated "
          "(individual-country sum is under 90% of the official World total):")
    display(likely_truncated)
else:
    print("No combinations look truncated — individual-country sums are consistent with official totals.")

10 country/category/year combinations look truncated (individual-country sum is under 90% of the official World total):


,Country,Category,Year,CIF_Value_USD_summed,CIF_Value_USD_world_total,pct_of_official_total
113,Colombia,General_auto_parts,2025,2.083921e+08,6.555958e+08,31.8
14,Argentina,General_auto_parts,2025,1.918527e+09,5.778835e+09,33.2
148,Ecuador,Lithium_battery,2025,4.527798e+07,1.172906e+08,38.6
13,Argentina,General_auto_parts,2024,2.354256e+09,5.634788e+09,41.8
134,Ecuador,General_auto_parts,2021,7.006689e+08,1.527978e+09,45.9
12,Argentina,General_auto_parts,2023,3.898320e+09,7.626258e+09,51.1
11,Argentina,General_auto_parts,2022,3.422018e+09,6.604543e+09,51.8
10,Argentina,General_auto_parts,2021,3.897935e+09,5.865305e+09,66.5
137,Ecuador,General_auto_parts,2024,3.300207e+08,4.940324e+08,66.8
112,Colombia,General_auto_parts,2023,3.398013e+08,4.376746e+08,77.6


In [10]:
save_processed(df_comtrade, "comtrade_imports_latam.csv")
save_processed(df_world_totals, "comtrade_world_totals_latam.csv")

16:58:30 | WARNING | Deduplicated: removed 6539 exact-duplicate rows (25.2%)
16:58:30 | INFO | Saved -> data/processed/comtrade_imports_latam.csv (19442 rows, 10 cols)
16:58:30 | WARNING | Deduplicated: removed 874 exact-duplicate rows (29.1%)
16:58:30 | INFO | Saved -> data/processed/comtrade_world_totals_latam.csv (2133 rows, 7 cols)


PosixPath('/Users/davidanampa/Documents/3. Data Analyst projects/latam-automotive-analytics/data/processed/comtrade_world_totals_latam.csv')

## Findings

China's share of the region's EV import value rose from **15% in 2021 to 86% in 2025**, while
total EV import value across the 8 countries grew roughly **19x** in the same period ($429M to
$8.0B). This is the clearest quantitative signal in the whole project: the "Chinese brands
gaining ground" pattern referenced in the original project brief is not just visible in the
press, it shows up directly in customs data.

**Limitation to keep in mind:** 13 country/category/year combinations were flagged as likely
truncated by the free-tier 500-row cap (see the check in step 5b) — all in `General_auto_parts`
and, to a lesser extent, `Lithium_battery` and `Combustion_pickup`. None fall in `EV_100pct`,
so the China-share figures above are not affected. Argentina's auto-parts totals in particular
should be treated as a lower bound, not a precise figure.

**Business opportunity note (a hypothesis to explore further, not a conclusion from this data
alone):** Mexico's dominant position in lithium-battery imports (see notebook 04) is best
explained by its role as a vehicle-assembly and export platform for the US market, not by
domestic EV demand — Mexico's own EV share (7%, notebook 01) is lower than Colombia's or
Brazil's. That raises a genuine question worth investigating outside this dataset: could a
similar battery-distribution or assembly role emerge in South America, anchored in a country
with the right conditions — available land, port access, energy costs, trade agreements — the
way Mexico's role is anchored in USMCA? Peru's Pacific port access and existing mining sector
(including lithium-adjacent minerals) would be a reasonable starting point for that separate
investigation, but this project's data does not test that hypothesis directly.